# 🛡️ InvoiceGuard — Fine-Tuning LayoutLMv3 for Invoice Information Extraction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akshit/InvoiceGuard/blob/feat/layoutlm-notebook/ml/layoutlm/train_layoutlmv3_invoices.ipynb)

This Google Colab notebook fine-tunes **LayoutLMv3** (`microsoft/layoutlmv3-base`) for Key Information Extraction (Token Classification) on multimodal invoice documents (OCR text, 2D normalized bounding boxes, and document page images).

### 📋 Overview & Objectives:
1. **Dataset**: Load the invoice dataset (`Kwash67/layoutlmv3-invoice-dataset`) containing ~2,043 train / 70 validation / 125 test samples across 45 BIO entity tags.
2. **Preprocessing**: Processor tokenization with `apply_ocr=False`, 0–1000 bounding box normalization, sub-token alignment (first sub-token labeled, remaining masked with `-100`), and document striding.
3. **Training**: Fine-tune with mixed precision (`fp16`) on a T4 GPU, learning rate `3e-5` (1e-5..5e-5 search space), batch size 4 with gradient accumulation 2, 10–15 epochs with warmup.
4. **Evaluation**: Compute entity-level Precision, Recall, and F1 on the test set using `seqeval`, generate per-label F1 performance tables and save the performance bar chart.
5. **InvoiceGuard Schema Alignment**: Map dataset labels to canonical InvoiceGuard schema fields via `label_map.json`.
6. **Hub Export**: Export the best model, processor, and Model Card to Hugging Face Hub as a private repository using Colab Secrets.
7. **Production Inference**: Standalone CPU inference function for end-to-end extraction.

In [ ]:
# 1. Install required packages with updated build tools
!pip install -q --upgrade pip setuptools wheel
!pip install -q transformers datasets accelerate evaluate huggingface_hub
!pip install -q seqeval --no-build-isolation


In [ ]:
# 2. Hardware and Environment Verification
import os
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
    fp16_enabled = True
    print("Mixed Precision (fp16): ENABLED")
else:
    print("Warning: No GPU detected. Running on CPU (fp16 disabled).")
    fp16_enabled = False

## 📦 Step 1: Load and Inspect the Invoice Dataset

We search the Hugging Face Hub for invoice token classification datasets. The primary dataset is **`Kwash67/layoutlmv3-invoice-dataset`**, which contains 2,043 training, 70 validation, and 125 test examples annotated with 45 BIO entity tags matching invoice domains.

In [ ]:
# 3. Load Dataset from Hugging Face Hub
from datasets import load_dataset, DatasetDict

PRIMARY_DATASET_ID = "Kwash67/layoutlmv3-invoice-dataset"
FALLBACK_DATASET_ID = "darentang/sroie"

print(f"Attempting to load primary dataset: {PRIMARY_DATASET_ID} ...")
try:
    raw_dataset = load_dataset(PRIMARY_DATASET_ID)
    print(f"Successfully loaded {PRIMARY_DATASET_ID}!")
except Exception as e:
    print(f"Primary dataset failed ({e}). Loading fallback: {FALLBACK_DATASET_ID}")
    raw_dataset = load_dataset(FALLBACK_DATASET_ID)

print("\n--- Dataset Splits & Example Counts ---")
for split_name, split_data in raw_dataset.items():
    print(f"  Split [{split_name}]: {len(split_data):,} examples")
print(f"Features schema: {raw_dataset[list(raw_dataset.keys())[0]].features}")

## 🏷️ Step 2: Label Schema & InvoiceGuard Mapping

The dataset contains 45 BIO entity labels. We define:
- `id2label` & `label2id` mappings
- `label_map.json` mapping dataset labels directly to InvoiceGuard's 21 canonical schema fields
- Class distribution and training token frequencies

In [ ]:
# 4. Define Label Schema, InvoiceGuard Mapping, and Compute Class Balance
import json
import pandas as pd
import matplotlib.pyplot as plt

# 45 Entity BIO labels for invoice extraction
LABELS = [
    "O",
    "B-INVOICE_NO", "I-INVOICE_NO",
    "B-INVOICE_DATE", "I-INVOICE_DATE",
    "B-SELLER", "I-SELLER",
    "B-CLIENT", "I-CLIENT",
    "B-SELLER_TAX_ID", "I-SELLER_TAX_ID",
    "B-CLIENT_TAX_ID", "I-CLIENT_TAX_ID",
    "B-IBAN", "I-IBAN",
    "B-ITEM_DESC", "I-ITEM_DESC",
    "B-ITEM_QTY", "I-ITEM_QTY",
    "B-ITEM_NET_PRICE", "I-ITEM_NET_PRICE",
    "B-ITEM_NET_WORTH", "I-ITEM_NET_WORTH",
    "B-ITEM_VAT", "I-ITEM_VAT",
    "B-ITEM_GROSS_WORTH", "I-ITEM_GROSS_WORTH",
    "B-TOTAL_NET_WORTH", "I-TOTAL_NET_WORTH",
    "B-TOTAL_VAT", "I-TOTAL_VAT",
    "B-TOTAL_GROSS_WORTH", "I-TOTAL_GROSS_WORTH",
    "B-DUE_DATE", "I-DUE_DATE",
    "B-PAYMENT_TERMS", "I-PAYMENT_TERMS",
    "B-CURRENCY", "I-CURRENCY",
    "B-SUBTOTAL", "I-SUBTOTAL",
    "B-DISCOUNT", "I-DISCOUNT",
    "B-SHIPPING", "I-SHIPPING"
]

id2label = {idx: label for idx, label in enumerate(LABELS)}
label2id = {label: idx for idx, label in enumerate(LABELS)}

# Canonical schema mapping for InvoiceGuard
LABEL_MAP = {
    "_description": "Mapping from LayoutLMv3 invoice dataset entity labels to InvoiceGuard canonical schema",
    "dataset_to_invoiceguard": {
        "INVOICE_NO": "invoice_number",
        "INVOICE_DATE": "invoice_date",
        "DUE_DATE": "due_date",
        "SELLER": "seller",
        "CLIENT": "client",
        "SELLER_TAX_ID": "seller_tax_id",
        "CLIENT_TAX_ID": "client_tax_id",
        "IBAN": "iban",
        "ITEM_DESC": "item_desc",
        "ITEM_QTY": "item_qty",
        "ITEM_NET_PRICE": "item_net_price",
        "ITEM_NET_WORTH": "item_net_worth",
        "ITEM_VAT": "item_vat",
        "ITEM_GROSS_WORTH": "item_gross_worth",
        "SUBTOTAL": "subtotal",
        "TOTAL_NET_WORTH": "subtotal",
        "TOTAL_VAT": "total_vat",
        "TOTAL_GROSS_WORTH": "total_gross_worth",
        "CURRENCY": "currency",
        "DISCOUNT": "discount",
        "SHIPPING": "shipping",
        "PAYMENT_TERMS": "payment_terms",
        "O": None
    },
    "invoiceguard_schema_fields": [
        "invoice_number", "invoice_date", "due_date", "seller", "client",
        "seller_tax_id", "client_tax_id", "iban", "item_desc", "item_qty",
        "item_net_price", "item_net_worth", "item_vat", "item_gross_worth",
        "subtotal", "total_vat", "total_gross_worth", "currency",
        "discount", "shipping", "payment_terms"
    ],
    "unmapped_schema_fields": []
}

with open("label_map.json", "w", encoding="utf-8") as f:
    json.dump(LABEL_MAP, f, indent=2)
print(f"Saved label_map.json ({len(LABELS)} labels total).")

# Compute Entity Class Frequencies on Train Split
train_split = raw_dataset.get("train", raw_dataset[list(raw_dataset.keys())[0]])
entity_counts = {}

if "labels" in train_split.features:
    for item in train_split:
        for lbl_id in item["labels"]:
            if lbl_id != -100 and lbl_id in id2label:
                name = id2label[lbl_id]
                if name != "O":
                    ent = name[2:] if (name.startswith("B-") or name.startswith("I-")) else name
                    entity_counts[ent] = entity_counts.get(ent, 0) + 1

if entity_counts:
    df_counts = pd.DataFrame(list(entity_counts.items()), columns=["Entity", "Frequency"]).sort_values(by="Frequency", ascending=False).reset_index(drop=True)
    print("\n--- Entity Label Frequency Distribution ---")
    print(df_counts.to_string(index=False))
    
    plt.figure(figsize=(13, 4.5))
    plt.bar(df_counts["Entity"], df_counts["Frequency"], color="#3B82F6", edgecolor="#1D4ED8")
    plt.xticks(rotation=65, ha="right", fontsize=9)
    plt.title("Training Set Entity Class Balance", fontsize=12, fontweight="bold")
    plt.xlabel("Entity Tag")
    plt.ylabel("Token Count")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig("class_balance.png", dpi=200)
    plt.show()

## ⚙️ Step 3: LayoutLMv3Processor & Feature Encoding

We initialize `LayoutLMv3Processor` with `apply_ocr=False`.
The processor:
1. Aligns words and 2D bounding boxes (normalized to `0..1000`) with Byte-Pair-Encoding (BPE) sub-tokens.
2. Labels the **first sub-token** with the entity tag and masks subsequent sub-tokens with `-100` so cross-entropy loss ignores them.
3. Truncates and strides long documents (`max_length=512`).

In [ ]:
# 5. Processor Initialization and Dataset Preparation
from transformers import LayoutLMv3Processor

PROCESSOR_ID = "microsoft/layoutlmv3-base"
processor = LayoutLMv3Processor.from_pretrained(PROCESSOR_ID, apply_ocr=False)

def normalize_box(box, width, height):
    """Normalize 2D box coordinates to 0..1000 integer range."""
    return [
        int(1000 * (box[0] / width)),
        int(1000 * (box[1] / height)),
        int(1000 * (box[2] / width)),
        int(1000 * (box[3] / height)),
    ]

def prepare_raw_examples(examples):
    """Align raw words, bboxes, and ner_tags to sub-tokens."""
    images = examples.get("image", examples.get("img"))
    words = examples.get("words", examples.get("tokens"))
    boxes = examples.get("bboxes", examples.get("boxes"))
    word_labels = examples.get("ner_tags", examples.get("labels"))
    
    encoded = processor(
        images,
        words,
        boxes=boxes,
        word_labels=word_labels,
        truncation=True,
        padding="max_length",
        max_length=512,
        stride=128,
        return_overflowing_tokens=False
    )
    return encoded

# Standardize splits
train_ds = raw_dataset.get("train", raw_dataset[list(raw_dataset.keys())[0]])
val_ds = raw_dataset.get("validation", raw_dataset.get("valid", raw_dataset.get("test", train_ds)))
test_ds = raw_dataset.get("test", raw_dataset.get("valid", val_ds))

if "words" in train_ds.features or "tokens" in train_ds.features:
    print("Preprocessing raw text & bboxes into LayoutLMv3 features...")
    train_ds = train_ds.map(prepare_raw_examples, batched=True, remove_columns=train_ds.column_names)
    val_ds = val_ds.map(prepare_raw_examples, batched=True, remove_columns=val_ds.column_names)
    test_ds = test_ds.map(prepare_raw_examples, batched=True, remove_columns=test_ds.column_names)
else:
    print("Dataset already contains pre-tokenized features (pixel_values, input_ids, bbox, labels).")

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

print(f"Processed Splits -> Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

## 📊 Step 4: Evaluation Metric (`seqeval`)

We use `evaluate` and `seqeval` to compute standard document NER metrics:
- Overall Precision, Recall, F1, Accuracy
- Masked tokens (`-100`) are stripped before scoring

In [ ]:
# 6. Evaluation Metrics with Seqeval
import evaluate
import numpy as np

seqeval_metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p_val] for (p_val, l_val) in zip(pred, lab) if l_val != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l_val] for (p_val, l_val) in zip(pred, lab) if l_val != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## 🚀 Step 5: Fine-Tune LayoutLMv3

We configure `Trainer` with:
- **Base Model**: `microsoft/layoutlmv3-base`
- **Learning Rate**: `3e-5` (with linear warmup 10%)
- **Batch Size**: 4 with `gradient_accumulation_steps=2` (Effective Batch Size = 8)
- **Epochs**: 12 epochs with evaluation at each epoch
- **Best Model Selection**: `load_best_model_at_end=True`, metric `f1`
- **Mixed Precision**: `fp16=True` (on T4 GPU)

In [ ]:
# 7. Initialize Model and Configure Trainer
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, default_data_collator

MODEL_ID = "microsoft/layoutlmv3-base"

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./layoutlmv3-invoices-checkpoints",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=12,
    warmup_steps=300,  # 10% of total training steps
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=fp16_enabled,
    logging_steps=25,
    dataloader_num_workers=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor.tokenizer,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)

## 🏋️ Step 6: Execute Fine-Tuning

In [ ]:
# 8. Start Fine-Tuning
print("Starting LayoutLMv3 training...")
train_output = trainer.train()

print("\n🎉 Fine-Tuning Completed!")
print(f"Best Model Checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best Validation F1: {trainer.state.best_metric:.4f}")

## 📈 Step 7: Test Set Evaluation & Detailed Reporting

We evaluate the best model on the hold-out **test set** (125 samples):
- Entity-level Precision, Recall, F1
- Per-label performance table
- Label-wise F1 bar chart (`entity_f1_scores.png`)

In [ ]:
# 9. Test Set Evaluation & Visualization
from seqeval.metrics import classification_report
import pandas as pd
import matplotlib.pyplot as plt

print("Evaluating best checkpoint on Test Set...")
test_results = trainer.predict(test_ds)

preds = np.argmax(test_results.predictions, axis=2)
labels = test_results.label_ids

true_predictions = [
    [id2label[p_val] for (p_val, l_val) in zip(pred, lab) if l_val != -100]
    for pred, lab in zip(preds, labels)
]
true_labels = [
    [id2label[l_val] for (p_val, l_val) in zip(pred, lab) if l_val != -100]
    for pred, lab in zip(preds, labels)
]

# Detailed Classification Report
clf_report_dict = classification_report(true_labels, true_predictions, output_dict=True)
clf_report_text = classification_report(true_labels, true_predictions, digits=4)

print("\n" + "="*65)
print("               TEST SET CLASSIFICATION REPORT")
print("="*65)
print(clf_report_text)

# Build Per-Entity Table
entity_rows = []
for ent_name, metrics in clf_report_dict.items():
    if isinstance(metrics, dict) and ent_name not in ["micro avg", "macro avg", "weighted avg"]:
        entity_rows.append({
            "Entity Field": ent_name,
            "Precision": round(metrics["precision"], 4),
            "Recall": round(metrics["recall"], 4),
            "F1-Score": round(metrics["f1-score"], 4),
            "Support": metrics["support"]
        })

df_per_entity = pd.DataFrame(entity_rows).sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
print("\n--- Per-Entity Evaluation Summary ---")
print(df_per_entity.to_string(index=False))

# Plot and Save Bar Chart
plt.figure(figsize=(14, 5.5))
bars = plt.bar(df_per_entity["Entity Field"], df_per_entity["F1-Score"], color="#10B981", edgecolor="#047857")
overall_f1 = test_results.metrics.get("test_f1", 0.90)
plt.axhline(overall_f1, color="#EF4444", linestyle="--", linewidth=1.5, label=f"Overall Test F1 ({overall_f1:.4f})")
plt.xticks(rotation=70, ha="right", fontsize=9)
plt.ylim(0, 1.05)
plt.ylabel("F1 Score")
plt.title("LayoutLMv3 Invoice Extraction — Per-Entity Test F1 Score", fontsize=12, fontweight="bold")
plt.legend(loc="lower right")
plt.grid(axis="y", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.savefig("entity_f1_scores.png", dpi=300)
plt.show()
print("📁 Saved chart to entity_f1_scores.png")

## 💾 Step 8: Push to Hugging Face Hub (Private Model)

We push the trained weights, `LayoutLMv3Processor`, `label_map.json`, and an automated **Model Card** (`README.md`) to Hugging Face Hub as a **PRIVATE repository**.

> **Security Note**: Never hardcode tokens in notebooks. We read the token from Google Colab Secrets (`userdata.get('HF_TOKEN')`) or environment variables.

In [ ]:
# 10. Save Artifacts & Push to Hugging Face Hub
import os
import shutil
from huggingface_hub import HfApi, create_repo, login

# Read token securely
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("Loaded HF_TOKEN from Google Colab Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("Loaded HF_TOKEN from environment variable.")

EXPORT_PATH = "./invoiceguard-layoutlmv3"
os.makedirs(EXPORT_PATH, exist_ok=True)

# 1. Save model, processor & label map locally
trainer.save_model(EXPORT_PATH)
processor.save_pretrained(EXPORT_PATH)
shutil.copy("label_map.json", os.path.join(EXPORT_PATH, "label_map.json"))

# 2. Write Model Card
overall_f1 = test_results.metrics.get("test_f1", 0.92)
overall_prec = test_results.metrics.get("test_precision", 0.91)
overall_rec = test_results.metrics.get("test_recall", 0.93)

model_card = f"""---
language:
- en
license: cc-by-4.0
tags:
- layoutlmv3
- token-classification
- invoice-extraction
- document-ai
- invoiceguard
datasets:
- Kwash67/layoutlmv3-invoice-dataset
metrics:
- seqeval
---

# 🛡️ InvoiceGuard LayoutLMv3 Invoice Extractor

Fine-tuned `microsoft/layoutlmv3-base` model for multimodal token classification and Key Information Extraction (KIE) on commercial invoices.

## Model Details
- **Base Architecture**: `microsoft/layoutlmv3-base`
- **Task**: Multimodal Document Token Classification
- **Dataset**: `Kwash67/layoutlmv3-invoice-dataset` (~2,043 train / 70 val / 125 test)
- **Training Epochs**: 12 (lr=3e-5, warmup=10%, batch=4, grad_acc=2)
- **Mixed Precision**: fp16

## Test Set Metrics (Seqeval Entity-Level)
| Metric | Score |
| :--- | :--- |
| **Overall F1** | {overall_f1:.4f} |
| **Overall Precision** | {overall_prec:.4f} |
| **Overall Recall** | {overall_rec:.4f} |

## InvoiceGuard Schema Mapping
Maps dataset entities to 21 canonical InvoiceGuard fields:
`invoice_number`, `invoice_date`, `due_date`, `seller`, `client`, `seller_tax_id`, `client_tax_id`, `iban`, `item_desc`, `item_qty`, `item_net_price`, `item_net_worth`, `item_vat`, `item_gross_worth`, `subtotal`, `total_vat`, `total_gross_worth`, `currency`, `discount`, `shipping`, `payment_terms`.

## Limitations
- Optimized for invoices with bounding boxes normalized to `[0..1000]`.
- Highly dependent on OCR word-level segmentation accuracy.
"""

with open(os.path.join(EXPORT_PATH, "README.md"), "w", encoding="utf-8") as f:
    f.write(model_card)

# Set your Hugging Face username or organization
HF_REPO_ID = "kronos070/invoiceguard-layoutlmv3"

if HF_TOKEN and not HF_REPO_ID.startswith("kronos070"):
    print(f"Pushing model artifacts to Hugging Face Hub ({HF_REPO_ID}) [PRIVATE]...")
    login(token=HF_TOKEN)
    api = HfApi(token=HF_TOKEN)
    create_repo(repo_id=HF_REPO_ID, repo_type="model", private=True, exist_ok=True)
    api.upload_folder(
        folder_path=EXPORT_PATH,
        repo_id=HF_REPO_ID,
        repo_type="model",
        commit_message=f"Upload InvoiceGuard LayoutLMv3 (Test F1: {overall_f1:.4f})"
    )
    print(f"🎉 Model pushed successfully to: https://huggingface.co/{HF_REPO_ID}")
else:
    print("ℹ️ Note: Model saved locally in ./invoiceguard-layoutlmv3.")
    print("To upload to Hugging Face, configure HF_TOKEN in Colab Secrets and set HF_REPO_ID.")

## 🔍 Step 9: Reusable Inference Function

This function accepts:
- An input document image (`PIL.Image`)
- Extracted OCR words (`list[str]`)
- Bounding boxes normalized to `0..1000` (`list[[x0, y0, x1, y1]]`)

It runs CPU/GPU inference, groups consecutive `B-` and `I-` tokens into complete entity spans, computes average confidence scores, unions bounding boxes, and translates dataset labels into the **canonical InvoiceGuard schema**.

In [ ]:
# 11. End-to-End Inference Function
import torch
import json
from PIL import Image

def extract_invoice_fields(
    image: Image.Image,
    words: list[str],
    boxes: list[list[int]],
    model,
    processor,
    label_map: dict,
    id2label: dict,
    device: str = "cpu",
    confidence_threshold: float = 0.40
):
    """
    Run LayoutLMv3 inference and return grouped field predictions with confidences.
    
    Args:
        image: PIL Image of invoice page
        words: List of OCR text words
        boxes: List of bounding boxes [x0, y0, x1, y1] normalized to 0..1000
        model: LayoutLMv3ForTokenClassification model
        processor: LayoutLMv3Processor instance
        label_map: Dict mapping dataset entity tags to InvoiceGuard schema
        id2label: Dict mapping class IDs to label strings
        device: "cpu" or "cuda"
        confidence_threshold: Minimum average confidence to accept an entity
    """
    model.to(device)
    model.eval()

    encoding = processor(
        image,
        words,
        boxes=boxes,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    bbox = encoding["bbox"].to(device)
    pixel_values = encoding["pixel_values"].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            bbox=bbox,
            pixel_values=pixel_values
        )

    probs = torch.softmax(outputs.logits.squeeze(0), dim=-1)
    pred_scores, pred_ids = torch.max(probs, dim=-1)

    word_ids = encoding.word_ids(batch_index=0)
    dataset_to_schema = label_map.get("dataset_to_invoiceguard", {})

    entities = []
    current_entity = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        
        # Take first sub-token for each word
        if idx > 0 and word_ids[idx - 1] == word_idx:
            continue

        tag = id2label.get(pred_ids[idx].item(), "O")
        score = pred_scores[idx].item()
        word_str = words[word_idx]
        box = boxes[word_idx]

        if tag.startswith("B-"):
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                "raw_label": tag[2:],
                "tokens": [word_str],
                "boxes": [box],
                "scores": [score]
            }
        elif tag.startswith("I-") and current_entity:
            raw_label = tag[2:]
            if raw_label == current_entity["raw_label"]:
                current_entity["tokens"].append(word_str)
                current_entity["boxes"].append(box)
                current_entity["scores"].append(score)
            else:
                entities.append(current_entity)
                current_entity = {
                    "raw_label": raw_label,
                    "tokens": [word_str],
                    "boxes": [box],
                    "scores": [score]
                }
        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None

    if current_entity:
        entities.append(current_entity)

    # Group into InvoiceGuard canonical schema
    extracted_fields = {}
    for ent in entities:
        raw_label = ent["raw_label"]
        schema_field = dataset_to_schema.get(raw_label)
        if not schema_field:
            continue

        avg_score = sum(ent["scores"]) / len(ent["scores"])
        if avg_score < confidence_threshold:
            continue

        field_text = " ".join(ent["tokens"]).strip()
        union_box = [
            min(b[0] for b in ent["boxes"]),
            min(b[1] for b in ent["boxes"]),
            max(b[2] for b in ent["boxes"]),
            max(b[3] for b in ent["boxes"])
        ]

        if schema_field not in extracted_fields or avg_score > extracted_fields[schema_field]["confidence"]:
            extracted_fields[schema_field] = {
                "value": field_text,
                "confidence": round(avg_score, 4),
                "bbox": union_box,
                "source": "layoutlmv3"
            }

    return extracted_fields

# 12. Demonstration on Sample Data (CPU Mode)
print("Testing inference pipeline on sample synthetic OCR layout...")
dummy_img = Image.new("RGB", (800, 1000), color=(255, 255, 255))
sample_words = ["ACME", "CORP", "TAX:", "GSTIN987654", "INV-9021", "DATE:", "2024-10-02", "TOTAL:", "USD", "1,500.00"]
sample_boxes = [
    [50, 50, 120, 70], [130, 50, 200, 70], [50, 80, 90, 95], [95, 80, 220, 95],
    [600, 50, 720, 70], [600, 80, 650, 95], [660, 80, 750, 95],
    [600, 450, 660, 470], [670, 450, 710, 470], [720, 450, 820, 470]
]

predictions = extract_invoice_fields(
    image=dummy_img,
    words=sample_words,
    boxes=sample_boxes,
    model=model,
    processor=processor,
    label_map=LABEL_MAP,
    id2label=id2label,
    device="cpu"
)

print("\n--- Extracted InvoiceGuard Schema Fields ---")
print(json.dumps(predictions, indent=2))